In [115]:
import sys
import importlib

sys.path.append('../utils')
import visualization
importlib.reload(visualization)
from visualization import save_to_html

import pandas as pd
import plotly.express as px
import itertools
from scipy.stats import ttest_ind, pointbiserialr

df = pd.read_parquet('../data/processed/telco_churn.parquet')

display(df.head())
display(df.info())
display(df.isnull().sum())

,customerID,gender,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,MonthlyCharges,TotalCharges,Churn,tenure_group,customer_type,avg_monthly_per_tenure_month,fiber_optic_flag,electronic_check_flag,month_to_month_flag,SeniorCitizen
0,7639-LIAYI,Male,0,0,52,1,Yes,DSL,Yes,No,...,79.75,4217.8,0,37+ months,Existing,1.53,0,0,0,0
1,9803-FTJCG,Male,1,1,71,1,Yes,DSL,Yes,Yes,...,66.85,4748.7,0,37+ months,Existing,0.94,0,0,0,0
2,5160-UXJED,Male,0,1,17,1,No,DSL,No,No,...,44.60,681.4,0,13-24 months,Existing,2.62,0,0,0,0
3,6122-EFVKN,Male,0,1,24,0,No phone service,DSL,Yes,No,...,35.75,830.8,0,13-24 months,Existing,1.49,0,0,0,0
4,3583-EKAPL,Male,0,0,1,1,No,DSL,No,No,...,55.00,55.0,1,1-6 months,Existing,55.00,0,1,1,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 27 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   customerID                    7043 non-null   string  
 1   gender                        7043 non-null   category
 2   Partner                       7043 non-null   int8    
 3   Dependents                    7043 non-null   int8    
 4   tenure                        7043 non-null   int64   
 5   PhoneService                  7043 non-null   int8    
 6   MultipleLines                 7043 non-null   category
 7   InternetService               7043 non-null   category
 8   OnlineSecurity                7043 non-null   category
 9   OnlineBackup                  7043 non-null   category
 10  DeviceProtection              7043 non-null   category
 11  TechSupport                   7043 non-null   category
 12  StreamingTV                   7043 non-null   ca

None

customerID                       0
gender                           0
Partner                          0
Dependents                       0
tenure                           0
PhoneService                     0
MultipleLines                    0
InternetService                  0
OnlineSecurity                   0
OnlineBackup                     0
DeviceProtection                 0
TechSupport                      0
StreamingTV                      0
StreamingMovies                  0
Contract                         0
PaperlessBilling                 0
PaymentMethod                    0
MonthlyCharges                   0
TotalCharges                     0
Churn                            0
tenure_group                     0
customer_type                    0
avg_monthly_per_tenure_month    11
fiber_optic_flag                 0
electronic_check_flag            0
month_to_month_flag              0
SeniorCitizen                    0
dtype: int64

In [116]:
# Univariate Analysis
# Распределение целевой переменной Churn

univariate_figs = {}

total_customers = df['Churn'].count()
churn_yes = (df['Churn'] == 1).sum()
churn_no = (df['Churn'] == 0).sum()

fig = px.pie(df, names='Churn', title='Распределение оттока клиентов (Churn)',
             color_discrete_sequence=px.colors.qualitative.Set2,
             hole=0.4)
fig.update_traces(textinfo='percent+label')
fig.update_layout(height=500)

print(f"Всего клиентов: {total_customers}")
print(f"Оставшихся (Churn = 1): {churn_no} - {round(churn_no / total_customers * 100.0, 2)}%") 
print(f"Ушедших (Churn = 0): {churn_yes} - {round(churn_yes / total_customers * 100.0, 2)}%")
fig.show() 

univariate_figs["Распределение Churn"] = fig

Всего клиентов: 7043
Оставшихся (Churn = 1): 5174 - 73.46%
Ушедших (Churn = 0): 1869 - 26.54%


In [117]:
# Общее распределение числовых переменных

numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

for col in numerical_cols:
    fig = px.histogram(df, x=col,
                        marginal='box',
                        nbins=50,
                        title=f'Общее распределение {col}',
                        color_discrete_sequence=['blue'],
                        barmode='overlay')
    fig.update_layout(height=500)

    print(f'Описание {col}:')
    display(df[col].describe())
    fig.show()

    univariate_figs[f"Распределение {col}"] = fig

Описание tenure:


count    7043.000000
mean       32.371149
std        24.559481
min         0.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64

Описание MonthlyCharges:


count    7043.000000
mean       64.761692
std        30.090047
min        18.250000
25%        35.500000
50%        70.350000
75%        89.850000
max       118.750000
Name: MonthlyCharges, dtype: float64

Описание TotalCharges:


count    7043.000000
mean     2279.734304
std      2266.794470
min         0.000000
25%       398.550000
50%      1394.550000
75%      3786.600000
max      8684.800000
Name: TotalCharges, dtype: float64

In [118]:
# Таблицы и графики частот для категориальных признаков

categorical_cols = [col for col in df.columns if col not in numerical_cols 
                    and col not in ['customerID', 'Churn', 'customer_type', 
                                    'avg_monthly_per_tenure_month', 'fiber_optic_flag', 
                                    'electronic_check_flag', 'month_to_month_flag']]

for col in categorical_cols:
    freq = df[col].value_counts().head(10)
    percent = df[col].value_counts(normalize=True).head(10) * 100
    table = pd.concat([freq, percent.round(2)], axis=1, keys=['Count', 'Percent (%)'])

    fig = px.pie(df, names=col, title=f'Распределение {col}',
             color_discrete_sequence=px.colors.qualitative.Set2,
             hole=0.4)
    fig.update_traces(textinfo='percent+label')
    fig.update_layout(height=500)

    print(f'Распределение {col}:')
    display(table)
    fig.show()

    univariate_figs[f"Распределение {col}"] = fig

save_to_html(
    figures_dict=univariate_figs,
    filename="eda_univariate.html",
    title="Univariate Analysis",
    dashboard_dir="eda"
)

Распределение gender:


,Count,Percent (%)
gender,,
Male,3555,50.48
Female,3488,49.52


Распределение Partner:


,Count,Percent (%)
Partner,,
0,3641,51.7
1,3402,48.3


Распределение Dependents:


...

Распределение tenure_group:


,Count,Percent (%)
tenure_group,,
37+ months,3001,42.61
1-6 months,1470,20.87
13-24 months,1024,14.54
25-36 months,832,11.81
7-12 months,705,10.01
New customer,11,0.16


Распределение SeniorCitizen:


,Count,Percent (%)
SeniorCitizen,,
0,5901,83.79
1,1142,16.21


'Сохранено в ../dashboards/eda/eda_univariate.html'

In [119]:
# Bivariate Analysis
# Распределение числовых переменных с разделением по Churn

bivariate_figs = {}
bivariate_tables = {}

for col in numerical_cols:
    fig = px.histogram(df, x=col, color='Churn', 
                        marginal='box',
                        nbins=50,
                        title=f'Распределение {col} по Churn',
                        color_discrete_sequence=px.colors.qualitative.Set2,
                        barmode='overlay')
    fig.update_layout(height=500)

    print(f'Описание {col} по Churn:')
    table = df.groupby('Churn')[col].describe().reset_index()
    display(table)
    fig.show()

    bivariate_figs[f"Распределение {col} по Churn"] = fig
    bivariate_tables[f"Описание {col} по Churn"] = table

Описание tenure по Churn:


,Churn,count,mean,std,min,25%,50%,75%,max
0,0,5174.0,37.569965,24.113777,0.0,15.0,38.0,61.0,72.0
1,1,1869.0,17.979133,19.531123,1.0,2.0,10.0,29.0,72.0


Описание MonthlyCharges по Churn:


,Churn,count,mean,std,min,25%,50%,75%,max
0,0,5174.0,61.265124,31.092648,18.25,25.10,64.425,88.4,118.75
1,1,1869.0,74.441332,24.666053,18.85,56.15,79.650,94.2,118.35


Описание TotalCharges по Churn:


,Churn,count,mean,std,min,25%,50%,75%,max
0,0,5174.0,2549.911442,2329.954215,0.00,572.9,1679.525,4262.85,8672.45
1,1,1869.0,1531.796094,1890.822994,18.85,134.5,703.550,2331.30,8684.80


In [120]:
# Статистические тесты: влияние числовых признаков на Churn

results = []

for col in numerical_cols:
    group_yes = df[df['Churn'] == 1][col]
    group_no  = df[df['Churn'] == 0][col]
    
    # t-тест
    t_stat, p_value_t = ttest_ind(group_yes, group_no, equal_var=False, nan_policy='omit')
    
    # Point-Biserial Correlation
    corr, p_value_pb = pointbiserialr(df['Churn'], df[col])
    
    # Интерпретация силы связи
    if abs(corr) < 0.1:
        strength = "очень слабая"
    elif abs(corr) < 0.3:
        strength = "слабая"
    elif abs(corr) < 0.5:
        strength = "средняя"
    else:
        strength = "сильная"
    
    results.append({
        'Признак': col,
        'Mean (No Churn)': round(group_no.mean(), 2),
        'Mean (Yes Churn)': round(group_yes.mean(), 2),
        'Разница средних': round(group_yes.mean() - group_no.mean(), 2),
        't-статистика': round(t_stat, 4),
        'p-value (t-test)': f"{p_value_t:.6f}",
        'Корреляция (Point-Biserial)': round(corr, 4),
        'Сила связи': strength,
        'p-value (корр.)': f"{p_value_pb:.6f}"
    })

# Вывод результатов в виде таблицы
print("Статистические тесты влияния числовых признаков на отток")
stats_df = pd.DataFrame(results)
display(stats_df)

bivariate_tables["Статистические тесты влияния числовых признаков на отток"] = stats_df

Статистические тесты влияния числовых признаков на отток


,Признак,Mean (No Churn),Mean (Yes Churn),Разница средних,t-статистика,p-value (t-test),Корреляция (Point-Biserial),Сила связи,p-value (корр.)
0,tenure,37.57,17.98,-19.59,-34.8238,0.000000,-0.3522,средняя,0.000000
1,MonthlyCharges,61.27,74.44,13.18,18.4075,0.000000,0.1934,слабая,0.000000
2,TotalCharges,2549.91,1531.80,-1018.12,-18.7066,0.000000,-0.1983,слабая,0.000000


In [121]:
# Анализ оттока (Churn = 1) по категориям

results = []

for col in categorical_cols:
    is_numeric = False
    if pd.api.types.is_numeric_dtype(df[col]):
        is_numeric = True
        df[col] = df[col].astype('category')

    churn_rate = (df.groupby(col, observed=False)['Churn']
                    .mean()
                    .mul(100)           
                    .round(2)
                    .reset_index(name='Churn_Rate'))
    
    churn_rate = churn_rate.sort_values('Churn_Rate', ascending=False)
    churn_rate_display = churn_rate[[col, 'Churn_Rate']]
    
    fig = px.bar(
        churn_rate,
        x=col,
        y='Churn_Rate',
        title=f'Процент оттока (Churn = 1) по {col}',
        color='Churn_Rate',                    
        color_continuous_scale='RdYlGn_r',       
        text='Churn_Rate',
        labels={'Churn_Rate': 'Процент ушедших (%)', col: col}        
    )
    
    fig.update_traces(texttemplate='%{y:.2f}%')
    
    print(f"\nChurn Rate по {col}:")
    display(churn_rate_display.style.hide(axis="index"))
    fig.show()

    bivariate_figs[f"Churn Rate по {col}"] = fig

    for _, row in churn_rate_display.iterrows():
            results.append({
                'category': col,
                'value': f"{row[col]}",
                'churn_rate': row['Churn_Rate']
            })

    if is_numeric:
        df[col] = df[col].astype('int8')

top_category = pd.DataFrame(results)
top10 = top_category.sort_values('churn_rate', ascending=False).head(10).reset_index(drop=True)

display(top10[['category', 'value', 'churn_rate']].style.format({'churn_rate': '{:.2f}%'})
        .set_caption("ТОП-10 самых рискованных категориальных признаков")) 

bivariate_tables["ТОП-10 самых рискованных категориальных признаков"] = top10[['category', 'value', 'churn_rate']].style.format({'churn_rate': '{:.2f}%'})

save_to_html(
     figures_dict=bivariate_figs,
     tables_dict=bivariate_tables,
     filename="eda_bivariate.html",
     title="Bivariate Analysis",
     dashboard_dir="eda")



Churn Rate по gender:


gender,Churn_Rate
Female,26.920000
Male,26.160000



Churn Rate по Partner:


Partner,Churn_Rate
0,32.960000
1,19.660000



Churn Rate по Dependents:


...

tenure_group,Churn_Rate
1-6 months,53.330000
7-12 months,35.890000
13-24 months,28.710000
25-36 months,21.630000
37+ months,11.930000
New customer,0.000000



Churn Rate по SeniorCitizen:


SeniorCitizen,Churn_Rate
1,41.680000
0,23.610000


,category,value,churn_rate
0,tenure_group,1-6 months,53.33%
1,PaymentMethod,Electronic check,45.29%
2,Contract,Month-to-month,42.71%
3,InternetService,Fiber optic,41.89%
4,OnlineSecurity,No,41.77%
5,SeniorCitizen,1.0,41.68%
6,TechSupport,No,41.64%
7,OnlineBackup,No,39.93%
8,DeviceProtection,No,39.13%
9,tenure_group,7-12 months,35.89%


'Сохранено в ../dashboards/eda/eda_bivariate.html'

In [122]:
# Multivariate Analysis
# Анализ комбинаций MonthlyCharges + Категориальные признаки

multivariate_figs = {}
multivariate_tables = {}

cat_cols = ['Contract', 'PaymentMethod', 'InternetService', 'TechSupport', 
                  'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'SeniorCitizen', 'tenure_group']
results = []

df['MonthlyCharges_bin'] = pd.qcut(df['MonthlyCharges'], q=4, duplicates='drop', 
                                labels=['Low', 'Medium-Low', 'Medium-High', 'High'])  

for cat_col in cat_cols:
    rate = (df.groupby([cat_col, 'MonthlyCharges_bin'], observed=False)['Churn'].mean()                 
        .mul(100)                  
        .round(2)
        .reset_index(name='Churn_Rate'))
    
    if cat_col != 'tenure_group':
        for _, row in rate.iterrows():
            results.append({
                'combination': f"{row[cat_col]} x {row['MonthlyCharges_bin']}",
                'feature_pair': f"{cat_col} x MonthlyCharges",
                'churn_rate': row['Churn_Rate']
            })

    
    if cat_col == 'tenure_group':
        rate = rate.sort_values('Churn_Rate', ascending=False).reset_index(drop=True)

        fig = px.bar(rate.head(20), 
                    x=cat_col, 
                    y='Churn_Rate', 
                    color='MonthlyCharges_bin',
                    barmode='group',
                    title=f'Churn Rate: {cat_col} x Уровень MonthlyCharges',
                    text='Churn_Rate',
                    labels={'MonthlyCharges_bin': 'Уровень MonthlyCharges'})
        
        fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
        fig.update_layout(height=550, xaxis_tickangle=45)
        fig.show()

        multivariate_figs[f"Churn Rate по {cat_col} x MonthlyCharges"] = fig

        table = rate.head(10).style.format({'Churn_Rate': '{:.2f}%'})
        multivariate_tables[f"ТОП-10 рискованных комбинаций tenure_group x MonthlyCharges"] = rate.head(10).style.format({'Churn_Rate': '{:.2f}%'})

        display(rate.head(10).style.format({'Churn_Rate': '{:.2f}%'})
                .set_caption(f"ТОП-10 рискованных комбинаций tenure_group x MonthlyCharges"))

top_combinations = pd.DataFrame(results)
top10 = top_combinations.sort_values('churn_rate', ascending=False).head(10).reset_index(drop=True)
multivariate_tables["ТОП-10 рискованных комбинаций c MonthlyCharges (исключая tenure_group)"] = top10[['feature_pair', 'combination', 'churn_rate']].style.format({'churn_rate': '{:.2f}%'})

display(top10[['feature_pair', 'combination', 'churn_rate']].style.format({'churn_rate': '{:.2f}%'})
        .set_caption("ТОП-10 рискованных комбинаций c MonthlyCharges (исключая tenure_group)"))


df.drop(columns=['MonthlyCharges_bin'], inplace=True, errors='ignore')

,tenure_group,MonthlyCharges_bin,Churn_Rate
0,1-6 months,High,77.50%
1,7-12 months,High,74.00%
2,1-6 months,Medium-High,72.48%
3,7-12 months,Medium-High,54.23%
4,13-24 months,High,54.12%
5,1-6 months,Medium-Low,52.81%
6,13-24 months,Medium-High,40.29%
7,25-36 months,High,38.39%
8,25-36 months,Medium-High,31.72%
9,1-6 months,Low,29.89%


,feature_pair,combination,churn_rate
0,InternetService x MonthlyCharges,Fiber optic x Medium-Low,59.01%
1,PaymentMethod x MonthlyCharges,Electronic check x Medium-High,55.00%
2,Contract x MonthlyCharges,Month-to-month x Medium-High,53.28%
3,Contract x MonthlyCharges,Month-to-month x High,52.18%
4,InternetService x MonthlyCharges,Fiber optic x Medium-High,50.94%
5,SeniorCitizen x MonthlyCharges,1 x Medium-High,49.87%
6,TechSupport x MonthlyCharges,No x Medium-High,49.87%
7,OnlineSecurity x MonthlyCharges,No x Medium-High,49.21%
8,DeviceProtection x MonthlyCharges,No x Medium-High,48.33%
9,OnlineBackup x MonthlyCharges,No x Medium-High,46.90%


In [124]:
# Multivariate Analysis
# Анализ комбинаций категориальных признаков

combinations = list(itertools.combinations(cat_cols, 2))
results_with_tenure = []
results_no_tenure = []

for col1, col2 in combinations:
    rate = (df.groupby([col1, col2], observed=False)['Churn']
        .mean()                 
        .mul(100)                  
        .round(2)
        .reset_index(name='Churn_Rate'))
    
    rate = rate.sort_values('Churn_Rate', ascending=False)
    
    
    if col1 == 'tenure_group' or col2 == 'tenure_group':
        for _, row in rate.iterrows():
            results_with_tenure.append({
                'col1' : col1,
                'col2' : col2,
                'feature_pair' : f"{col1} x {col2}",
                'combination': f"{row[col1]} x {row[col2]}",
                'churn_rate': row['Churn_Rate']
            })

    if col1 != 'tenure_group' and col2 != 'tenure_group':
        for _, row in rate.iterrows():
            results_no_tenure.append({
                'col1' : col1,
                'col2' : col2,
                'feature_pair' : f"{col1} x {col2}",
                'combination': f"{row[col1]} x {row[col2]}",
                'churn_rate': row['Churn_Rate']
            })


top_combinations_with_tenure = pd.DataFrame(results_with_tenure)
top10_with_tenure = top_combinations_with_tenure.sort_values('churn_rate', ascending=False).head(10).reset_index(drop=True)
multivariate_tables["ТОП-10 самых рискованных комбинаций (включая tenure_group)"] = top10_with_tenure[['feature_pair', 'combination', 'churn_rate']].style.format({'churn_rate': '{:.2f}%'})

display(top10_with_tenure[['feature_pair', 'combination', 'churn_rate']].style.format({'churn_rate': '{:.2f}%'})
        .set_caption("ТОП-10 самых рискованных комбинаций (включая tenure_group)"))

top_combinations_no_tenure = pd.DataFrame(results_no_tenure)
top10_no_tenure = top_combinations_no_tenure.sort_values('churn_rate', ascending=False).head(10).reset_index(drop=True)
multivariate_tables["ТОП-10 самых рискованных комбинаций (исключая tenure_group)"] = top10_no_tenure[['feature_pair', 'combination', 'churn_rate']].style.format({'churn_rate': '{:.2f}%'})

display(top10_no_tenure[['feature_pair', 'combination', 'churn_rate']].style.format({'churn_rate': '{:.2f}%'})
        .set_caption("ТОП-10 самых рискованных комбинаций (исключая tenure_group)"))

save_to_html(
    figures_dict=multivariate_figs,
    tables_dict=multivariate_tables,
    filename="eda_multivariate.html",
    title="Multivariate Analysis",
    dashboard_dir="eda"
)


,feature_pair,combination,churn_rate
0,InternetService x tenure_group,Fiber optic x 1-6 months,74.19%
1,SeniorCitizen x tenure_group,1 x 1-6 months,72.81%
2,PaymentMethod x tenure_group,Electronic check x 1-6 months,67.20%
3,OnlineSecurity x tenure_group,No x 1-6 months,66.23%
4,DeviceProtection x tenure_group,Yes x 1-6 months,66.10%
5,TechSupport x tenure_group,No x 1-6 months,65.63%
6,OnlineBackup x tenure_group,No x 1-6 months,64.26%
7,DeviceProtection x tenure_group,No x 1-6 months,62.13%
8,InternetService x tenure_group,Fiber optic x 7-12 months,61.06%
9,SeniorCitizen x tenure_group,1 x 7-12 months,57.28%


,feature_pair,combination,churn_rate
0,Contract x SeniorCitizen,Month-to-month x 1,54.65%
1,Contract x InternetService,Month-to-month x Fiber optic,54.61%
2,PaymentMethod x OnlineBackup,Electronic check x No,53.92%
3,Contract x PaymentMethod,Month-to-month x Electronic check,53.73%
4,PaymentMethod x SeniorCitizen,Electronic check x 1,53.37%
5,PaymentMethod x InternetService,Electronic check x Fiber optic,53.23%
6,PaymentMethod x TechSupport,Electronic check x No,53.18%
7,PaymentMethod x OnlineSecurity,Electronic check x No,53.17%
8,OnlineBackup x SeniorCitizen,No x 1,52.77%
9,PaymentMethod x DeviceProtection,Electronic check x No,52.06%


'Сохранено в ../dashboards/eda/eda_multivariate.html'

### **Key Insights & Business Recommendations**

#### 1. **Общий уровень оттока**
Общий Churn Rate составляет **26.54%** (ушли 1869 клиентов из 7043).

#### 2. **Доминирующий фактор — короткий стаж пользования**
- Клиенты в первые **1-6 месяцев** имеют churn rate **53.33%**.
- В комбинации с другими факторами этот показатель поднимается ещё выше (до **74.19%**).

#### 3. **Топ факторов оттока**

##### Самые рискованные одиночные признаки:
- `PaymentMethod = 'Electronic check'` — **45.29%**
- `Contract = 'Month-to-month'` — **42.71%**
- `InternetService = 'Fiber optic'` — **41.89%**
- `No TechSupport` — **41.64%**
- `No OnlineSecurity` — **41.77%**
- `SeniorCitizen = Yes` — **41.68%**

##### Самые рискованные комбинации (с учётом tenure_group):
- `1-6 months + Fiber optic` — **74.19%**
- `1-6 months + SeniorCitizen` — **72.81%**
- `1-6 months + Electronic check` — **67.20%**
- `1-6 months + No OnlineSecurity` — **66.23%**
- `1-6 months + No TechSupport` — **65.63%**

##### Самые рискованные комбинации среди клиентов (без учёта tenure_group):
- `Month-to-month + SeniorCitizen` — **54.65%**
- `Month-to-month + Fiber optic` — **54.61%**
- `Electronic check + No OnlineBackup` — **53.92%**
- `Month-to-month + Electronic check` — **53.73%**

#### 4. **Анализ числовых признаков**

Корреляционный анализ числовых переменных показал следующие результаты:

- **`tenure`** (стаж): Point-Biserial correlation = **-0,3522** (средняя отрицательная связь). Чем дольше клиент пользуется услугами, тем ниже вероятность оттока.
- **`MonthlyCharges`** (ежемесячный платёж): Point-Biserial correlation = **+0,1934** (слабая положительная связь). Более высокий платёж умеренно повышает риск оттока.
- **`TotalCharges`** (общая сумма платежей): Point-Biserial correlation = **-0,1983** (слабая отрицательная связь).

Хотя `MonthlyCharges` демонстрирует относительно слабую прямую корреляцию, его влияние значительно усиливается при сочетании с определёнными категориальными факторами:

**Самые рискованные комбинации с MonthlyCharges:**
- `1-6 months + High MonthlyCharges` — **77.50%**
- `7-12 months + High MonthlyCharges` — **74.00%**
- `1-6 months + Medium-High MonthlyCharges` — **72.48%**
- `Fiber optic + Medium-Low MonthlyCharges` — **59.01%**
- `Electronic check + Medium-High MonthlyCharges` — **55.00%**

#### 4. **Слабые факторы**
- Пол клиента (`gender`) практически не влияет на отток (разница — 0.76%).
- Наличие/отсутствие `PhoneService` имеет минимальное влияние (разница — 1.78%).

#### 5. **Основные гипотезы**
- Основной причиной оттока является **неудовлетворённость ценой и гибкостью контракта**.
- Клиенты, использующие дорогие и современные услуги (Fiber optic), ожидают высокого качества, которого, возможно, не получают.
- Отсутствие дополнительных услуг (особенно киберзащиты и технической поддержки) снижает лояльность клиентов.

### **Рекомендации для бизнеса**
- Разработать специальные программы удержания для новых клиентов (первые 6 месяцев).
- Пересмотреть условия оплаты через Electronic check или предложить бонусы за другие способы оплаты.
- Стимулировать переход клиентов с помесячного контракта на годовые/двухгодовые.
- Улучшить качество поддержки (TechSupport) и киберзащиты (OnlineSecurity).
- Проведение отдельного анализа качества и ценовой политики услуги **Fiber optic**.